# Phase 3: Data Preprocessing and Standardization

## Step 1: Missing Value Analysis

The first step in data preprocessing is to examine the structured dataset for missing values. Missing data can negatively affect data quality and model performance. Therefore, the number and percentage of missing values are calculated for each feature to determine the appropriate preprocessing strategy, such as imputation or retaining missing values where clinically appropriate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/prepared_dataset.csv")

# Display the first five rows
df.head()

,subject_id,stay_id,hadm_id,intime,outtime,gender,race,arrival_transport,disposition,temperature,...,oral_medications,consults_count,procedure_count,age,resources_used,tiragecase,Expert 1 Opinion,Expert 2 Opinion,Expert 3 Opinion,Final Decision
0,12867223,32373901,25038197.0,2121-01-12 17:47:00,2121-01-12 22:43:00,M,WHITE,AMBULANCE,ADMITTED,NaN,...,2,0,0,79.032146,6,79-year-old white male with a history of depre...,DISAGREE,DISAGREE,NaN,REMOVE
1,12482649,30078245,24501616.0,2143-03-21 07:46:00,2143-03-21 15:04:00,M,WHITE,AMBULANCE,ADMITTED,97.3,...,0,0,0,52.218019,5,A 45-year-old white male with a history of int...,DISAGREE,DISAGREE,NaN,REMOVE
2,18530850,37186254,25480845.0,2167-02-12 23:52:00,2167-02-13 17:15:00,M,WHITE,AMBULANCE,HOME,96.6,...,4,0,0,62.117052,4,A 61-year-old male presented to the emergency ...,AGREE,AGREE,NaN,RETAIN
3,10320289,30019253,27936912.0,2161-06-21 07:53:00,2161-06-21 11:51:00,M,WHITE,AMBULANCE,ADMITTED,NaN,...,1,0,1,53.469082,4,A 53-year-old white male with a history of COP...,DISAGREE,UNCERTAIN,DISAGREE,REMOVE
4,16307504,31731173,23603634.0,2158-10-24 03:35:00,2158-10-24 08:58:00,M,WHITE,AMBULANCE,ADMITTED,NaN,...,0,0,1,63.810830,2,A 63-year-old male with a history of aortic re...,AGREE,AGREE,NaN,RETAIN


In [ ]:
# Check the shape of the dataset
print("Dataset Shape:", df.shape)

# Count missing values
missing_count = df.isnull().sum()

# Calculate missing value percentage
missing_percentage = (missing_count / len(df)) * 100

# Create a summary table
missing_summary = (
    pd.DataFrame({
        'Missing Values': missing_count,
        'Percentage (%)': missing_percentage
    })
    .sort_values(by='Percentage (%)', ascending=False)
)

# Display only columns with missing values
missing_summary = missing_summary[missing_summary['Missing Values'] > 0]

print("Missing Value Summary:")
display(missing_summary)

Dataset Shape: (50, 57)
Missing Value Summary:


,Missing Values,Percentage (%)
Expert 3 Opinion,34,68.0
temperature,17,34.0
resprate,14,28.0
dbp,14,28.0
o2sat,13,26.0
sbp,13,26.0
heartrate,12,24.0
hadm_id,10,20.0
pain,5,10.0


## Step 2: Missing Value Handling

Before imputing missing values, the `pain` column was examined because it contained both numeric pain scores and non-numeric clinical entries such as `Critical`, `crit`, `UA`, `uta`, and `unable`. These text values do not represent valid numerical pain scores and were therefore converted to missing values (`NaN`).

Subsequently, median imputation was applied to the numerical clinical variables, including temperature, heart rate, respiratory rate, oxygen saturation, systolic blood pressure, diastolic blood pressure, and pain score. Median imputation was selected because it is robust to outliers commonly observed in emergency department data.

Identifier columns (`hadm_id`) and expert annotation columns (`Expert 3 Opinion`) were left unchanged since they cannot be reliably inferred.

In [ ]:
# Create a copy of the dataset
df_clean = df.copy()

# Clean the pain column

# Replace non-numeric pain entries with NaN
pain_invalid = ['Critical', 'crit', 'UA', 'uta', 'unable']

df_clean['pain'] = df_clean['pain'].replace(pain_invalid, pd.NA)

# Convert pain to numeric
df_clean['pain'] = pd.to_numeric(df_clean['pain'], errors='coerce')

# Median imputation

median_columns = [
    'temperature',
    'heartrate',
    'resprate',
    'o2sat',
    'sbp',
    'dbp',
    'pain'
]

for col in median_columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Verify missing values
print(df_clean[median_columns].isnull().sum())

temperature    0
heartrate      0
resprate       0
o2sat          0
sbp            0
dbp            0
pain           0
dtype: int64


## Step 3: Duplicate Record Detection

Duplicate records can introduce bias into data analysis and machine learning models by giving disproportionate importance to repeated observations. Therefore, the dataset was examined for duplicate entries. If duplicate records were identified, they would be removed to ensure that each patient encounter is represented only once.

In [ ]:
# Check for duplicate rows
duplicate_count = df_clean.duplicated().sum()

print(f"Number of duplicate records: {duplicate_count}")

# Remove duplicate rows if any exist
df_clean = df_clean.drop_duplicates()

print("Dataset Shape After Removing Duplicates:", df_clean.shape)

Number of duplicate records: 0
Dataset Shape After Removing Duplicates: (50, 57)


## Step 4: Data Type Inspection and Correction

The dataset was inspected to verify that each feature had an appropriate data type. Correct data types are essential for accurate analysis and machine learning. Numerical variables should be stored as numeric types, while date and time variables should be converted to datetime format for time-based analysis and feature engineering.

In [ ]:
# Display the data type of each column
data_types = pd.DataFrame({
    'Column': df_clean.columns,
    'Data Type': df_clean.dtypes.values
})

display(data_types)

,Column,Data Type
0,subject_id,int64
1,stay_id,int64
2,hadm_id,float64
3,intime,object
4,outtime,object
5,gender,object
6,race,object
7,arrival_transport,object
8,disposition,object
9,temperature,float64


## Step 5: Date and Time Conversion

The `intime` and `outtime` columns were stored as text (`object`) data types. These columns were converted to the `datetime` format to enable time-based analysis and future feature engineering, such as calculating the patient's length of stay in the emergency department.

In [ ]:
# Convert date and time columns to datetime format
df_clean['intime'] = pd.to_datetime(df_clean['intime'])
df_clean['outtime'] = pd.to_datetime(df_clean['outtime'])

# Verify the updated data types
print(df_clean[['intime', 'outtime']].dtypes)

intime     datetime64[ns]
outtime    datetime64[ns]
dtype: object


## Step 6: Categorical Variable Consistency Check

Categorical variables were examined to identify inconsistencies such as variations in spelling, capitalization, or formatting. Maintaining consistent categorical values is important for reliable data analysis and prevents the creation of duplicate categories during later encoding. Since the dataset is curated, this step primarily serves as a quality assurance check.

In [ ]:
# List of categorical columns
categorical_columns = [
    'gender',
    'race',
    'arrival_transport',
    'disposition',
    'Expert 1 Opinion',
    'Expert 2 Opinion',
    'Expert 3 Opinion',
    'Final Decision'
]

# Display unique values for each categorical column
for col in categorical_columns:
    print(f"\n{col}")
    print("-" * len(col))
    print(sorted(df_clean[col].dropna().unique()))


gender
------
['F', 'M']

race
----
['ASIAN', 'ASIAN - CHINESE', 'BLACK/AFRICAN AMERICAN', 'HISPANIC/LATINO - HONDURAN', 'PATIENT DECLINED TO ANSWER', 'UNKNOWN', 'WHITE', 'WHITE - OTHER EUROPEAN', 'WHITE - RUSSIAN']

arrival_transport
-----------------
['AMBULANCE', 'UNKNOWN', 'WALK IN']

disposition
-----------
['ADMITTED', 'HOME']

Expert 1 Opinion
----------------
['AGREE', 'DISAGREE', 'UNCERTAIN']

Expert 2 Opinion
----------------
['AGREE', 'DISAGREE', 'UNCERTAIN']

Expert 3 Opinion
----------------
['AGREE', 'DISAGREE']

Final Decision
--------------
['REMOVE', 'RETAIN']


### Results

The categorical variables were inspected for inconsistencies in naming, spelling, and capitalization. All categories were found to be consistently formatted, and no further standardization was required. Therefore, the original categorical values were retained to preserve the integrity of the dataset.

In [ ]:
print("Categorical variables are already standardized.")
print("No modifications were required.")

Categorical variables are already standardized.
No modifications were required.


## Step 7: Save the Cleaned Dataset

After completing data cleaning and quality improvement, the cleaned dataset was saved for use in the next preprocessing stage, which involves feature preparation and data transformation for machine learning and Large Language Model (LLM) integration.

In [ ]:
# Save the cleaned dataset
df_clean.to_csv("cleaned_dataset.csv", index=False)

print("Cleaned dataset saved successfully.")
print("Final Dataset Shape:", df_clean.shape)

Cleaned dataset saved successfully.
Final Dataset Shape: (50, 57)


In [ ]:
from google.colab import files

files.download("cleaned_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")

##Step 8: feature engineering

In this step, new predictive features were derived from the existing clinical data to provide richer context for the machine learning models. First, the patient's Emergency Department (ED) Length of Stay was calculated in hours by subtracting the intime timestamp from the outtime timestamp. Second, continuous age values *were* categorized into discrete demographic brackets (Pediatric, Young Adult, Adult, Older Adult, Senior) to help the model identify risk patterns associated with specific life stages.

In [ ]:
#calculating length of stay
df['ED_Length_of_Stay_Hours'] = (df_clean['outtime'] - df_clean['intime']).dt.total_seconds() / 3600

#categerizing age group
bins = [0, 18, 35, 50, 65, 120]
labels = ['Pediatric', 'Young Adult', 'Adult', 'Older Adult', 'Senior']
df['Age_Group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)
print("Feature Engineering complete: Added Length of Stay and Age Groups.")

Feature Engineering complete: Added Length of Stay and Age Groups.


##Step 9:Encoding category variables

Computers can't read words like 'Ambulance' or 'Walk-in'. So, we created a new column called 'Ambulance' and just put a 1 if they arrived in one, and a 0 if they didn't. For the final diagnosis, we just turned words into simple numbers (e.g., Discharge = 0, Admit = 1).

In [ ]:
from sklearn.preprocessing import LabelEncoder

#encoding for these categericol variabes only
nominal_cols = ['gender', 'race', 'arrival_transport', 'Age_Group']

#encoing 0/1 only for the existing columns
nominal_cols = [col for col in nominal_cols if col in df.columns]
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

#these are for final ouptuts
le = LabelEncoder()
target_cols = ['disposition', 'Final Decision']
label_mappings = {}

for col in target_cols:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))
        # Store the mapping ('Discharge' = 0, 'Admit' = 1) for the documentation report
        label_mappings[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print("\n--- ENCODING MAPPINGS FOR DOCUMENTATION ---")
for target, mapping in label_mappings.items():
    print(f"{target}: {mapping}")




--- ENCODING MAPPINGS FOR DOCUMENTATION ---
disposition: {'ADMITTED': np.int64(0), 'HOME': np.int64(1)}
Final Decision: {'REMOVE': np.int64(0), 'RETAIN': np.int64(1)}


##Step 10: Numerical Feature Scaling

To ensure that features with naturally larger numerical ranges (like Systolic Blood Pressure) do not disproportionately dominate the model's learning process over features with smaller ranges (like Respiration Rate), a StandardScaler was applied. This process normalized all continuous physiological and temporal variables so that they possess a mean of 0 and a standard deviation of 1, effectively leveling the mathematical playing field.

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

#
scaler = StandardScaler()
continuous_features = [
    'age', 'temperature', 'heartrate', 'resprate',
    'o2sat', 'sbp', 'dbp', 'pain', 'resources_used', 'ED_Length_of_Stay_Hours'
]

#only scaling exist in the the table
cols_to_scale = [col for col in continuous_features if col in df.columns]
for col in cols_to_scale:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[cols_to_scale] = df[cols_to_scale].fillna(df[cols_to_scale].median())

df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
print("\nNumerical features standardized.")


Numerical features standardized.


In [ ]:
df = df.drop(columns=['intime', 'outtime'], errors='ignore')
df = df.drop(columns=['subject_id', 'stay_id', 'hadm_id'])
output_path = '/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/preprocessed_dataset.csv'
df.to_csv(output_path, index=False)
print(f"\nPart B Complete! ML-ready dataset saved to: {output_path}")


Part B Complete! ML-ready dataset saved to: /content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/preprocessed_dataset.csv


In [ ]:
# Load the saved preprocessed dataset
preprocessed_df = pd.read_csv(output_path)

# Display basic information
print("Preprocessed Dataset Shape:", preprocessed_df.shape)

# Display the first 5 rows
display(preprocessed_df.head())

Preprocessed Dataset Shape: (50, 65)


,disposition,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,acuity,chiefcomplaint,...,race_UNKNOWN,race_WHITE,race_WHITE - OTHER EUROPEAN,race_WHITE - RUSSIAN,arrival_transport_UNKNOWN,arrival_transport_WALK IN,Age_Group_Young Adult,Age_Group_Adult,Age_Group_Older Adult,Age_Group_Senior
0,0,0.014517,-1.052148,-0.583438,-0.084039,1.264178,-0.446499,0.735316,1,"s/p Fall, Head injury",...,False,True,False,False,False,False,False,False,False,True
1,0,-1.146856,-0.939739,0.018044,-1.484685,0.267641,0.227462,0.210090,2,"Head lac, s/p Fall",...,False,True,False,False,False,False,False,False,True,False
2,1,-2.163058,-0.096672,-0.583438,-0.550921,2.118353,-0.151641,1.260542,2,Epigastric pain,...,False,True,False,False,False,False,False,False,True,False
3,0,0.014517,-0.096672,0.018044,0.382843,-0.064538,-0.151641,0.078784,1,"Transfer, Respiratory distress",...,False,True,False,False,False,False,False,False,True,False
4,0,0.014517,-0.096672,0.018044,0.382843,-0.064538,-0.151641,0.078784,1,Chest pain,...,False,True,False,False,False,False,False,False,True,False
